<a href="https://colab.research.google.com/github/ishwarikale44-byte/LP-3-Practical/blob/main/Copy_of_BT3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install web3 eth-tester py-evm py-solc-x

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.6/798.6 kB 14.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 586.3/586.3 kB 12.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 60.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 62.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25/25 [eth-tester]


In [ ]:
from web3 import Web3, EthereumTesterProvider
from solcx import compile_source, install_solc

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [ ]:
install_solc("0.8.20")

print("Solidity Compiler Installed")

Solidity Compiler Installed


In [ ]:
bank_code = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

contract BankAccount {

    uint public balance;

    function deposit() public payable {
        balance += msg.value;
    }

    function withdraw(uint amount) public {
        require(amount <= balance, "Insufficient Balance");

        balance -= amount;

        payable(msg.sender).transfer(amount);
    }

    function showBalance() public view returns(uint) {
        return balance;
    }
}
'''

print("Smart Contract Created Successfully")

Smart Contract Created Successfully


In [ ]:
compiled_sol = compile_source(
    bank_code,
    solc_version="0.8.20"
)

contract_interface = list(compiled_sol.values())[0]

bytecode = contract_interface['bin']
abi = contract_interface['abi']

print("Smart Contract Compiled Successfully")

Smart Contract Compiled Successfully


In [ ]:
w3 = Web3(EthereumTesterProvider())

account = w3.eth.accounts[0]

w3.eth.default_account = account

print("Blockchain Connected Successfully")
print("Account Address:", account)

Blockchain Connected Successfully
Account Address: 0x7E5F4552091A69125d5DfCb7b8C2659029395Bdf


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
Bank = w3.eth.contract(
    abi=abi,
    bytecode=bytecode
)

tx_hash = Bank.constructor().transact()

tx_receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

bank = w3.eth.contract(
    address=tx_receipt.contractAddress,
    abi=abi
)

print("Smart Contract Deployed Successfully")
print("Contract Address:", bank.address)

Smart Contract Deployed Successfully
Contract Address: 0xF2E246BB76DF876Cef8b38ae84130F4F55De395b


In [ ]:
deposit_amount = Web3.to_wei(5, 'ether')

tx = bank.functions.deposit().transact({
    'from': account,
    'value': deposit_amount
})

w3.eth.wait_for_transaction_receipt(tx)

print("5 Ether Deposited Successfully")

5 Ether Deposited Successfully


In [ ]:
balance = bank.functions.showBalance().call()

print("Current Bank Balance:", Web3.from_wei(balance, 'ether'), "ETH")

Current Bank Balance: 5 ETH


In [ ]:
withdraw_amount = Web3.to_wei(2, 'ether')

tx = bank.functions.withdraw(withdraw_amount).transact({
    'from': account
})

w3.eth.wait_for_transaction_receipt(tx)

print("2 Ether Withdrawn Successfully")

2 Ether Withdrawn Successfully


In [ ]:
balance = bank.functions.showBalance().call()

print("----- BANK ACCOUNT DETAILS -----")
print("Initial Deposit: 5 ETH")
print("Withdraw Amount: 2 ETH")
print("Final Balance:", Web3.from_wei(balance, 'ether'), "ETH")

----- BANK ACCOUNT DETAILS -----
Initial Deposit: 5 ETH
Withdraw Amount: 2 ETH
Final Balance: 3 ETH
